# Imports

In [ ]:
import pandas as pd
import plip_analysis as pa
from pathlib import Path
import plotly.express as px
import json

# Load the data

In [ ]:
data_path = Path("20241120_plip_analysis")

In [ ]:
datasets = {}
for virus in ["SARS-CoV-2", "MERS-CoV"]:
    datatype_dict = {}
    for data_type in ['crystal', 'docked']:
        dataset_name = f"{virus[:4]}_{data_type}"
        datatype_dict[data_type] = {csv_path.stem: pa.PLIntReport.from_csv(csv_path) for csv_path in data_path.glob(f"{dataset_name}*.csv")}
    datasets[virus] = datatype_dict

# convert names to be easier to process

In [ ]:
import re

In [ ]:
for virus, datatype_dict in datasets.items():
    for datatype, plint_dict in datatype_dict.items():
        names = [name for name in plint_dict.keys()]
        new_plint_dict = {}
        for k, value in plint_dict.items():
            asap_id = re.search(r'(ASAP-[0-9]*)', k).group(1)
            new_plint_dict[asap_id] = value
        datatype_dict[datatype] = new_plint_dict

# Turn into raw df

In [ ]:
hydrophobic = [pa.InteractionType.PiStacking, pa.InteractionType.HydrophobicInteraction]
hydrogen_bond = [pa.InteractionType.HydrogenBondDonor, pa.InteractionType.HydrogenBondAcceptor]
records = []
for virus, datatype_dict in datasets.items():
    for datatype, plint_dict in datatype_dict.items():
        for name, plint in plint_dict.items():
            records.append({'variant': virus, 
                            'datatype': datatype, 
                            'name': name, 
                            'n_interactions': len(plint.interactions), 
                            'n_hydrophobic': len([inter for inter in plint.interactions if inter.interaction_type == pa.InteractionType.HydrophobicInteraction]),
                            'n_hydrogen_bond': len([inter for inter in plint.interactions if inter.interaction_type in hydrogen_bond]),
                            'n_pi_stacking': len([inter for inter in plint.interactions if inter.interaction_type == pa.InteractionType.PiStacking]),
                            'n_salt_bridge': len([inter for inter in plint.interactions if inter.interaction_type == pa.InteractionType.SaltBridge]),
                            'n_halogen_bond': len([inter for inter in plint.interactions if inter.interaction_type == pa.InteractionType.HalogenBond]),
                            })

In [ ]:
raw_df = pd.DataFrame.from_records(records)

In [ ]:
raw_df.groupby(['variant', 'datatype']).count()

In [ ]:
# plint_example = raw_df.iloc[0]['plint']

In [ ]:
# for interaction in plint_example.interactions:
#     if interaction.interaction_type in hydrophobic:
#         print(interaction)

In [ ]:
count_df = raw_df.groupby(['variant', 'datatype']).count()['name'].reset_index()

In [ ]:
count_df.columns = ['variant', 'datatype', 'count']
count_df['count'] = count_df['count'].astype(int)

In [ ]:
count_df

In [ ]:
px.bar(count_df, x='variant', 
       color='datatype', 
       template='simple_white', 
       barmode='group', 
       height=400, 
       width=800, 
       y='count',)

# Get the number of interactions

In [ ]:
raw_df.groupby(['variant', 'datatype'])['name'].nunique()

In [ ]:
# filter the crystal structures by the docked structures
docked_mers = raw_df[(raw_df['datatype'] == 'docked') & (raw_df['variant'] == 'MERS-CoV')]
crystal_mers = raw_df[(raw_df['datatype'] == 'crystal') & (raw_df['variant'] == 'MERS-CoV')]
docked_sars = raw_df[(raw_df['datatype'] == 'docked') & (raw_df['variant'] == 'SARS-CoV-2')]
crystal_sars = raw_df[(raw_df['datatype'] == 'crystal') & (raw_df['variant'] == 'SARS-CoV-2')]

In [ ]:
comp_crystal_mers = crystal_mers[crystal_mers['name'].isin(docked_mers['name'])]
comp_crystal_sars = crystal_sars[crystal_sars['name'].isin(docked_sars['name'])]
comp_docked_mers = docked_mers[docked_mers['name'].isin(crystal_mers['name'])]
comp_docked_sars = docked_sars[docked_sars['name'].isin(crystal_sars['name'])]

In [ ]:
# Recombine
comparable_df = pd.concat([comp_crystal_mers, comp_crystal_sars, comp_docked_mers, comp_docked_sars])

In [ ]:
comparable_df.groupby(['variant', 'datatype'])['name'].nunique()

### try histogram

In [ ]:
fig = px.histogram(comparable_df, x="n_interactions", height=400, width=800, template='simple_white',color='datatype', barmode='overlay', barnorm=None, facet_col='variant')
fig.show()
# fig.update_layout(legend={'title': 'Similarity Metric'})
# fig.update_yaxes(title='Cumulative Probability')
# fig.update_xaxes(title='PLIF Recall')

# fig.write_image("cdf_fingerprint_comparison.png")

In [ ]:
fig = px.ecdf(comparable_df, 
              x="n_interactions", 
              height=400, 
              width=600, 
              template='simple_white',
              color='variant', 
              line_dash='datatype', 
              ecdfnorm=None)
fig.show()

In [ ]:
fig = px.ecdf(comparable_df, x="n_hydrophobic", height=400, width=800, template='simple_white',color='variant', line_dash='datatype', ecdfnorm=None)
fig.show()

In [ ]:
fig = px.ecdf(comparable_df, x="n_hydrogen_bond", height=400, width=800, template='simple_white',color='variant', line_dash='datatype', ecdfnorm=None)
fig.show()

# Get differences by interaction type

In [ ]:
# pivot to tidy dataframe
new_df = comparable_df.groupby(['variant', 'datatype', 'name']).sum().reset_index()
new_df = new_df.melt(id_vars=['variant', 'datatype'], value_vars=[#'n_interactions', 
                                                                  'n_hydrophobic', 'n_hydrogen_bond', 'n_pi_stacking', 'n_salt_bridge', 'n_halogen_bond'])

In [ ]:
# plot combined ecdf
fig = px.ecdf(new_df, 
              x="value", 
              height=1200, 
              width=800, 
              template='simple_white',
              # color="variant",
              color='variable', 
              line_dash='datatype',
              facet_row='variant',
              # facet_row='variable',
              )
fig.show()

In [ ]:
# get difference between datatypes

In [ ]:
px.bar(new_df, x='variable', 
       color='datatype', 
       template='simple_white', 
       barmode='group', 
       height=400, 
       width=800, 
       y='value',
       # log_y=True,
       facet_col='variant',
       text_auto=True,)

# Get fingerprint comparison

In [ ]:
score_list = []
missing = []
for virus, datatype_dict in datasets.items():
    for level in pa.FingerprintLevel:
        for name, docked_plint_report in datatype_dict["docked"].items():
            crystal_plint_report = datatype_dict["crystal"].get(name)
            if crystal_plint_report is None:
                missing.append(name)
                continue
            print(f"Processing {name} {virus} {level}")
            print(crystal_plint_report, docked_plint_report)
            score_list.append({'ASAP_Ligand_ID': name, 'Variant': virus, **pa.InteractionScore.from_fingerprints(crystal_plint_report, docked_plint_report, level).dict()})

In [ ]:
missing

In [ ]:
df = pd.DataFrame.from_records(score_list)

In [ ]:
df.groupby(["provenance", "Variant"]).count()

# Get cumulative distribution 

The ratio of the intersection is the tversky index or recall

In [ ]:
df["ratio_of_intersection"] = df["number_of_interactions_in_intersection"] / df["number_of_interactions_in_reference"]

the ratio of the query to intersection is also interesting 

In [ ]:
df['ratio_of_query'] = df["number_of_interactions_in_query"] / df["number_of_interactions_in_reference"]

In [ ]:
by_interaction_type = df[df["provenance"] == pa.FingerprintLevel.ByInteractionType.value]
by_everything = df[df["provenance"] == pa.FingerprintLevel.ByEverything.value]

In [ ]:
pa.FingerprintLevel._member_names_

In [ ]:
filtered = df[df["provenance"].isin(["ByTotalInteractions","ByInteractionType", "ByInteractionTypeAndAtomTypes", "ByEverything"])]

In [ ]:
# fig = px.ecdf(filtered, x="tversky_index", 
#               height=400, 
#               width=800, 
#               color='provenance', 
#               template='simple_white', 
#               line_dash="Variant",
#               ecdfnorm=None,
#               # facet_row='Variant'
#               )
fig = px.ecdf(filtered, x="tversky_index", 
              height=400, 
              width=1200, 
              # color='provenance',
              color='Variant',
              template='simple_white', 
              line_dash="Variant",
              ecdfnorm=None,
              # facet_row='provenance',
              facet_col='provenance',
              )
# fig.update_layout(legend={'title': 'Similarity Metric'})
fig.update_layout(legend={'title': 'Viral Variant'})
# fig.update_yaxes(title='Cumulative Probability')
fig.for_each_yaxis(lambda y: y.update(title=' '))
import re
fig.for_each_annotation(lambda x: x.update(text=re.sub(r"(\w)([A-Z])", r"\1 \2", x.text.replace('provenance=', ''))))

fig.update_layout(yaxis1=dict(title='Total Number of Structures'))
fig.update_xaxes(title='PLIF Recall')
fig.show()
fig.write_image("cdf_fingerprint_comparison.png")
fig.write_image("cdf_fingerprint_comparison.svg")

In [ ]:
ints_comparison_df = df.groupby(["provenance", "Variant"])[["number_of_interactions_in_query", "number_of_interactions_in_reference", "number_of_interactions_in_intersection"]].mean().reset_index()

In [ ]:
ints_comparison_df['recall'] = ints_comparison_df["number_of_interactions_in_intersection"] / ints_comparison_df["number_of_interactions_in_reference"]

In [ ]:
px.bar(ints_comparison_df.sort_values(by='recall'), x='provenance', 
       color='Variant', 
       template='simple_white', 
       barmode='group', 
       height=400, 
       width=800, 
       y='recall',)

In [ ]:
import plotly.graph_objects as go
import numpy as np
# Extract the data
data = by_interaction_type["tversky_index"]

# Create the histogram
fig = go.Figure()
fig.add_trace(go.Histogram(x=data, name='Histogram', histnorm='probability'))

# # Calculate the CDF
hist, bins = np.histogram(data, bins=30)
cdf = np.cumsum(hist) / len(data)
# 
# Create the CDF trace
fig.add_trace(go.Scatter(x=bins, y=cdf, name='CDF', mode='lines'))

# Update layout
fig.update_layout(title='PLIF Recall for MERS Predictions', height=400, width=600, template="simple_white")

# Update axis titles
fig.update_xaxes(title="PLIF Recall")
fig.update_yaxes(title="Probability")

# Show the plot
fig.show()

In [ ]:
fig.write_image("plif_recall_for_mers_predictions.png")

# Why is MERS doing better?

In [ ]:
stringent = df[df["provenance"] == "ByEverything"]

In [ ]:
stringent.sort_values(by="ASAP_Ligand_ID")

In [ ]:
stringent[stringent["ASAP_Ligand_ID"] == "ASAP-0000526"]

In [ ]:
example = raw_df[raw_df["name"] == "ASAP-0000526"]

In [ ]:
### pivot to tidy df
tidy_example = example.melt(id_vars=['variant', 'datatype', 'name'], value_vars=['n_interactions', 'n_hydrophobic', 'n_hydrogen_bond', 'n_pi_stacking', 'n_salt_bridge', 'n_halogen_bond'])

In [ ]:
tidy_example

In [ ]:
px.histogram(stringent, x='tversky_index', 
              height=400, 
              width=800, 
              color='Variant', 
              template='simple_white', 
              barmode='overlay', 
              barnorm=None)

In [ ]:
px.bar(tidy_example, x='variable', 
       color='datatype', 
       template='simple_white', 
       barmode='group', 
       height=400, 
       width=800, 
       y='value',
       facet_col='variant',
       text_auto=True,)

In [ ]:
good_example = stringent.sort_values(by="tversky_index", ascending=False).ASAP_Ligand_ID.to_list()[0]

In [ ]:
stringent[stringent["ASAP_Ligand_ID"] == good_example]

In [ ]:
def image_example(df, example_id):
    example = df[df["name"] == example_id]
    tidy_example = example.melt(id_vars=['variant', 'datatype', 'name'], value_vars=['n_interactions', 'n_hydrophobic', 'n_hydrogen_bond', 'n_pi_stacking', 'n_salt_bridge', 'n_halogen_bond'])
    fig = px.bar(tidy_example, x='variable', 
       color='datatype', 
       template='simple_white', 
       barmode='group', 
       height=400, 
       width=800, 
       y='value',
       facet_col='variant',
       text_auto=True,
                 title=f"Example {example_id}")
    return fig

In [ ]:
image_example(raw_df, good_example)

In [ ]:
bad_example = stringent.sort_values(by=["Variant", "tversky_index"], ascending=True).ASAP_Ligand_ID.to_list()[0]
stringent[stringent["ASAP_Ligand_ID"] == bad_example]

In [ ]:
image_example(raw_df, bad_example)